# Sports Card Extractor & eBay Search

Extract structured data from sports card images (Baseball, Football, Basketball, Hockey) using OpenAI's structured outputs (`response_format`), then search eBay.

This notebook demonstrates the same **schema-first extraction pattern** as the Pokemon card extractor, applied to a different domain.

## 1. Setup & Install

In [ ]:
!pip install openai python-dotenv pydantic pandas --quiet

In [ ]:
import os
import base64
import webbrowser
from urllib.parse import quote
from typing import Optional, List, Literal

import openai
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import display, Image, Markdown

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

## 2. Data Model (Pydantic)

A `SportsCard` model that captures the key fields for baseball, football, basketball, and hockey cards.

The same **structured outputs** pattern applies — define the schema, let the AI extract.

In [ ]:
class SportsCard(BaseModel):
    """Structured representation of a sports card extracted from an image."""

    # Core identification
    player_name: str = Field(description="Full name of the player (e.g. 'Mike Trout', 'Patrick Mahomes')")
    sport: Optional[str] = Field(default=None, description="Sport (e.g. 'Baseball', 'Football', 'Basketball', 'Hockey')")
    team: Optional[str] = Field(default=None, description="Team name (e.g. 'Los Angeles Angels', 'Kansas City Chiefs')")
    position: Optional[str] = Field(default=None, description="Player position (e.g. 'CF', 'QB', 'PG', 'C')")
    
    # Card details
    card_number: Optional[str] = Field(default=None, description="Card number (e.g. '1', '150', 'RC-1')")
    set_name: Optional[str] = Field(default=None, description="Set name (e.g. 'Topps Chrome', 'Panini Prizm', 'Upper Deck')")
    year: Optional[int] = Field(default=None, description="Year printed on the card or copyright year")
    manufacturer: Optional[str] = Field(default=None, description="Card manufacturer (e.g. 'Topps', 'Panini', 'Upper Deck', 'Bowman')")
    
    # Variants and parallels
    parallel: Optional[str] = Field(default=None, description="Parallel type (e.g. 'Refractor', 'Silver Prizm', 'Gold', 'Pink')")
    serial_number: Optional[str] = Field(default=None, description="Serial numbering if present (e.g. '25/99', '1/1')")
    is_rookie_card: Optional[bool] = Field(default=None, description="True if marked as a rookie card (RC logo or text)")
    is_autograph: Optional[bool] = Field(default=None, description="True if the card contains an autograph")
    is_memorabilia: Optional[bool] = Field(default=None, description="True if the card contains a jersey/patch/relic")
    
    # Grading (if visible)
    grading_company: Optional[str] = Field(default=None, description="Grading company if slabbed (e.g. 'PSA', 'BGS', 'SGC', 'CGC')")
    grade: Optional[str] = Field(default=None, description="Grade if visible (e.g. '10', '9.5', 'GEM MT 10')")
    
    # Additional info
    additional_notes: Optional[str] = Field(default=None, description="Insert names, special markings, errors, promo info")

    @property
    def display_name(self) -> str:
        parts = [self.player_name]
        if self.year:
            parts.append(f"({self.year})")
        if self.set_name:
            parts.append(f"- {self.set_name}")
        if self.parallel:
            parts.append(f"- {self.parallel}")
        if self.is_rookie_card:
            parts.append("RC")
        return " ".join(parts)

    @property
    def ebay_search_query(self) -> str:
        """Build a focused eBay search string from the structured fields."""
        tokens = []
        if self.year:
            tokens.append(str(self.year))
        if self.manufacturer:
            tokens.append(self.manufacturer)
        if self.set_name:
            tokens.append(self.set_name)
        tokens.append(self.player_name)
        if self.card_number:
            tokens.append(f"#{self.card_number}")
        if self.parallel:
            tokens.append(self.parallel)
        if self.is_rookie_card:
            tokens.append("RC")
        if self.serial_number:
            tokens.append(f"/{self.serial_number.split('/')[-1]}")
        if self.is_autograph:
            tokens.append("Auto")
        if self.grade:
            tokens.append(f"{self.grading_company or 'PSA'} {self.grade}")
        return " ".join(tokens)

    @property
    def ebay_url(self) -> str:
        return f"https://www.ebay.com/sch/i.html?_nkw={quote(self.ebay_search_query)}"

    def to_markdown_table(self) -> str:
        """Render card data as a markdown table for Jupyter display."""
        lines = ["| **Field** | **Value** |", "| :-- | :-- |"]
        for field_name, value in self.model_dump().items():
            if value is not None and value != [] and value != False:
                label = field_name.replace("_", " ").title()
                if isinstance(value, bool):
                    value = "Yes" if value else "No"
                if isinstance(value, list):
                    value = ", ".join(value)
                lines.append(f"| {label} | {value} |")
        return "\n".join(lines)

    def _repr_markdown_(self):
        """Rich display in Jupyter notebooks."""
        return f"### {self.display_name}\n\n{self.to_markdown_table()}"

## 3. Extraction Function

Uses `openai.beta.chat.completions.parse()` with `response_format=SportsCard`.  
Same pattern as Pokemon — the API returns a **parsed Pydantic object** directly.

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert sports card analyst and collector. Given an image of a sports card "
    "(baseball, football, basketball, hockey, or other sports), extract every piece of "
    "structured information you can identify. Be precise with player names, card numbers, "
    "set names, parallels, and serial numbering. Identify if the card is a rookie card (RC), "
    "autograph, or memorabilia/relic card. If the card is graded/slabbed, extract the grading "
    "company and grade. Use null for any field you cannot confidently determine from the image."
)


def build_image_content(image_source: str) -> dict:
    """Build the image content block for either a URL or a local file path."""
    if image_source.startswith(("http://", "https://")):
        return {
            "type": "image_url",
            "image_url": {"url": image_source},
        }
    else:
        with open(image_source, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        ext = os.path.splitext(image_source)[1].lower().lstrip(".")
        mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "webp": "webp"}.get(ext, "jpeg")
        return {
            "type": "image_url",
            "image_url": {"url": f"data:image/{mime};base64,{b64}"},
        }


def extract_card(image_source: str) -> SportsCard:
    """
    Send an image to OpenAI's vision model and return a validated SportsCard.

    Uses structured outputs (response_format) so the API returns a
    Pydantic-validated object directly — no JSON parsing required.
    """
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Extract all structured data from this sports card image."},
                    build_image_content(image_source),
                ],
            },
        ],
        response_format=SportsCard,
        temperature=0.1,
    )

    card = completion.choices[0].message.parsed

    if card is None:
        refusal = completion.choices[0].message.refusal
        raise ValueError(f"Extraction failed: {refusal or 'No structured output returned'}")

    return card

## 4. Extract a Card!

Try it with different sports cards. Examples below cover baseball, football, and basketball.

In [ ]:
# Mike Trout Topps Chrome rookie
IMAGE_URL = "https://i.ebayimg.com/images/g/fBQAAOSwbRVl8o0d/s-l1600.webp"

# Preview the image inline
display(Image(url=IMAGE_URL, width=300))

In [ ]:
card = extract_card(IMAGE_URL)

# Rich table display via _repr_markdown_
card

## 5. Inspect the Data

Since it's a Pydantic model, you get typed access to every field.

In [ ]:
# Access fields directly — fully typed, IDE-friendly
print(f"Player: {card.player_name}")
print(f"Sport: {card.sport}")
print(f"Team: {card.team}")
print(f"Year: {card.year}")
print(f"Set: {card.set_name}")
print(f"Rookie Card: {card.is_rookie_card}")
print(f"Parallel: {card.parallel}")
print(f"Grade: {card.grading_company} {card.grade}" if card.grade else "Grade: Raw")

In [ ]:
# Dump to dict or JSON — ready for storage, APIs, whatever
card.model_dump()

## 6. eBay Search

In [ ]:
print(f"Search query: {card.ebay_search_query}")
display(Markdown(f"**[Search eBay for this card]({card.ebay_url})**"))

In [ ]:
# Uncomment to open in your browser automatically
# webbrowser.open(card.ebay_url)

## 7. Batch Processing — Multiple Sports

Process cards from different sports → pandas DataFrame.

In [ ]:
# Mix of baseball, football, basketball
urls = [
    "https://i.ebayimg.com/images/g/fBQAAOSwbRVl8o0d/s-l1600.webp",  # Mike Trout
    "https://i.ebayimg.com/images/g/Z~QAAOSwmDRmCe4c/s-l1600.webp",  # Patrick Mahomes
    "https://i.ebayimg.com/images/g/3l8AAOSwZtVl8UkZ/s-l1600.webp",  # LeBron James
]

cards: list[SportsCard] = []
for url in urls:
    try:
        c = extract_card(url)
        cards.append(c)
        print(f"✓ {c.display_name}")
    except Exception as e:
        print(f"✗ Failed on {url}: {e}")

print(f"\nExtracted {len(cards)} card(s)")

In [ ]:
# Pydantic → DataFrame in one line
df = pd.DataFrame([c.model_dump() for c in cards])

# Add computed columns
df["ebay_query"] = [c.ebay_search_query for c in cards]
df["ebay_url"] = [c.ebay_url for c in cards]

display(df[["player_name", "sport", "team", "year", "set_name", "is_rookie_card", "parallel", "grade"]])

## 8. The Pattern Generalizes

This same approach works for:

- **Pokemon cards** → `PokemonCard` model
- **Sports cards** → `SportsCard` model (this notebook)
- **Magic: The Gathering** → `MTGCard` model
- **Yu-Gi-Oh** → `YuGiOhCard` model
- **Invoices** → `Invoice` model
- **Receipts** → `Receipt` model
- **Medical records** → `MedicalRecord` model

Define the schema. Pass it as `response_format`. Get structured data back.

No OCR. No regex. No fragile pipelines.